# Phase II: selection of most discriminative genes using IG

## Part 1: IG Calculation
- Further filter the gene pool by taking only those genes with IG higher than the mean IG of all genes.

In [ ]:
import pandas as pd

from sklearn.feature_selection import mutual_info_classif

In [ ]:
# Get class info from id maping to health status
health_stat = pd.read_csv("../data/GSE252168/id_to_health_status.tsv", sep= "\t", usecols=[1])
health_stat.head()

,Status
0,control
1,control
2,control
3,control
4,control


In [ ]:
# Get the class labels
y = health_stat["Status"]

In [ ]:
y

0      control
1      control
2      control
3      control
4      control
        ...   
298    control
299    control
300    control
301    control
302    control
Name: Status, Length: 303, dtype: str

In [ ]:
# Upload the expression matrix and prepare for the mutual_info_classif function
expression_df = pd.read_csv("../data/GSE252168/row_filtered.tsv", sep="\t")
expression_df.drop("ID_REF", axis=1, inplace=True)
expression_df.head()

,ILMN_1762337,ILMN_1736007,ILMN_2383229,ILMN_1814316,ILMN_1668111,ILMN_1755321,ILMN_1698554,ILMN_1760414,ILMN_2061446,ILMN_1668851,...,ILMN_1779783,ILMN_1685547,ILMN_2348512,ILMN_1743643,ILMN_1656676,ILMN_2371169,ILMN_1701875,ILMN_1786396,ILMN_1653618,ILMN_2137536
0,6.601397,6.555132,6.563006,6.538115,6.487908,6.502019,6.467476,6.593454,7.114948,6.344277,...,6.636482,6.493231,6.816312,7.070625,8.129110,9.825466,10.612416,7.642387,7.119752,7.073798
1,6.477026,6.640189,6.738270,6.554947,6.451093,6.499183,6.552973,6.554356,6.955625,6.395458,...,6.501612,6.540461,6.792972,7.196791,8.405085,9.505995,10.615032,7.826965,7.222498,6.656577
2,6.528409,6.500988,6.645548,6.461470,6.333161,6.616165,6.701698,6.446938,6.879105,6.454641,...,6.682851,6.710829,6.909647,7.253427,8.306038,9.562776,10.077904,7.853563,7.465050,6.800421
3,6.547416,6.509216,6.628310,6.569797,6.313699,6.506304,6.822026,6.600196,7.172154,6.450936,...,6.587851,6.581868,6.848492,7.400357,8.434583,10.452790,10.550622,7.974522,7.269712,6.603563
4,6.623571,6.498857,6.607876,6.574199,6.391719,6.490693,6.538222,6.563677,6.966809,6.430273,...,6.745439,6.529034,6.891276,7.013339,8.154735,9.235334,9.891232,7.617143,7.320744,7.174409


In [ ]:
# create the matrix for mutual_info_classif function
X = expression_df.values

In [ ]:
# Calculate IG for each gene and assign the expression matrix as dense by setting discrete_features to False
ig_results = mutual_info_classif(X, y, discrete_features=False)

In [ ]:
# Calculate the average IG for whole gene population
pop_mean = ig_results.mean()
print(f"The average IG for the gene pool population is {pop_mean:2f}")

The average IG for the gene pool population is 0.071058


In [ ]:
# Create a df to store IG values of the genes
ig_result_df = pd.DataFrame([ig_results], columns=expression_df.columns)
ig_result_df.head()

,ILMN_1762337,ILMN_1736007,ILMN_2383229,ILMN_1814316,ILMN_1668111,ILMN_1755321,ILMN_1698554,ILMN_1760414,ILMN_2061446,ILMN_1668851,...,ILMN_1779783,ILMN_1685547,ILMN_2348512,ILMN_1743643,ILMN_1656676,ILMN_2371169,ILMN_1701875,ILMN_1786396,ILMN_1653618,ILMN_2137536
0,0.058405,0.082485,0.045625,0.079398,0.176929,0.030019,0.013105,0.073541,0.064791,0.179462,...,0.024358,0.11743,0.16496,0.047723,0.083551,0.13124,0.224093,0.093285,0.221991,0.100968


In [ ]:
# Get the genes where their IG is higher than the average IG
filt_genes = ig_result_df.loc[:, ig_result_df.iloc[0]>=pop_mean]
filt_genes = filt_genes.transpose().rename(columns={0:"IG_value"})

In [ ]:
# Check how many genes filtered after IG 
print(f"The number of genes before IG filtration: {expression_df.shape[1]}")
print(f"The number of genes after IG filtration: {filt_genes.shape[0]}")

The number of genes before IG filtration: 14044
The number of genes after IG filtration: 5608


In [ ]:
# # Save the result 
# filt_genes.to_csv("../data/GSE252168/IG_results.tsv", sep="\t")

## Part 2: Map the selected genes to HGNC symbols

In [ ]:
# Upload the DEA result table to get gene symbols for corresponding gene IDs
dea_df = pd.read_csv("../data/GSE252168/dea_filtered.tsv", sep="\t", usecols=[0,1])
dea_df.head()

,ID,Gene.symbol
0,ILMN_1769409,TMEM261
1,ILMN_1762769,POLR2J4
2,ILMN_1750507,RPL9
3,ILMN_1754195,RPL31
4,ILMN_1787949,RPS15A


In [ ]:
# Create a dict where we have mapping from gene id to gene symbol
gene_id_to_symbol = dict(zip(dea_df["ID"], dea_df["Gene.symbol"]))
print(gene_id_to_symbol)

{'ILMN_1769409': 'TMEM261', 'ILMN_1762769': 'POLR2J4', 'ILMN_1750507': 'RPL9', 'ILMN_1754195': 'RPL31', 'ILMN_1787949': 'RPS15A', 'ILMN_3236713': 'SNHG1', 'ILMN_3245458': 'SNORA61', 'ILMN_2207539': 'RPS17', 'ILMN_1656807': 'RPL27', 'ILMN_2207533': 'RPS17', 'ILMN_1777378': 'COMMD6', 'ILMN_2369785': 'SNRPD2', 'ILMN_1659786': 'TRMT13', 'ILMN_1677239': 'CCDC14', 'ILMN_1681846': 'ZNF540', 'ILMN_3251662': 'HINT3', 'ILMN_2315569': 'N6AMT1', 'ILMN_1663160': 'ZNF337', 'ILMN_1737015': 'RPL39', 'ILMN_1807710': 'HINT1', 'ILMN_1676842': 'BTAF1', 'ILMN_2318725': 'EEF1B2', 'ILMN_1652594': 'ACTR5', 'ILMN_2383097': 'RPL17', 'ILMN_1712678': 'RPS27L', 'ILMN_1710078': 'TMEM181', 'ILMN_2145518': 'TMEM126B', 'ILMN_2192385': 'TTC19', 'ILMN_1806106': 'GNL3', 'ILMN_1776181': 'BIRC3', 'ILMN_1695362': 'ZNF32', 'ILMN_1733050': 'SEC31B', 'ILMN_1812478': 'ZNHIT3', 'ILMN_1694587': 'EEF1B2', 'ILMN_2119774': 'CYP2R1', 'ILMN_1755536': 'PFDN5', 'ILMN_1797425': 'DDX55', 'ILMN_1756360': 'RPL35A', 'ILMN_2087060': 'TOMM7', 

In [ ]:
# Rename the gene IDs to gene symbol using the dict 
filt_genes.rename(index=gene_id_to_symbol, inplace=True)
filt_genes.head()

,IG_value
A1BG,0.082485
RBFOX1,0.079398
A3GALT2,0.176929
AADAC,0.073541
AADACL4,0.179462


In [ ]:
# # Save the filtered genes with their IG values
# filt_genes.to_csv("../data/GSE252168/IG_results_v2", sep="\t")